# Restocking a shelf by value iteration

A corner shop restocks one product every morning. Every order pays a fixed fee on top of
the price of the units, so ordering a little every day is wasteful, and units left on the
shelf overnight cost money too. You will write the shop as an MDP in arrays, solve it with
value iteration, and read off the best order for every stock level.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

CAPACITY = 5                                        # the shelf holds at most 5 units
DEMAND_PROBS = np.array([0.1, 0.2, 0.3, 0.3, 0.1])  # P(demand = 0, 1, 2, 3, 4) on any day
PRICE, UNIT_COST, ORDER_FEE, HOLDING_COST = 6.0, 2.0, 6.0, 0.5
GAMMA = 0.9

## Part 1 (coding task): The shop as arrays

A day runs as follows.

1. In the morning the shelf holds $s$ units, $0 \le s \le 5$, and the shop orders $a$ units,
   $0 \le a \le 5$. The order arrives at once. Units that do not fit on the shelf are thrown
   away, but every unit ordered is paid for.
2. During the day $D$ customers each want one unit, with $D$ drawn from `DEMAND_PROBS`. A
   customer who finds the shelf empty buys nothing and does not come back.
3. The day's reward is `PRICE` per unit sold, minus `UNIT_COST` per unit ordered, minus
   `ORDER_FEE` if anything was ordered, minus `HOLDING_COST` per unit still on the shelf at the
   end of the day. Those units are the next morning's state.

Write `build_model(capacity, demand_probs, price, unit_cost, order_fee, holding_cost)`. It
returns `P` of shape `(capacity + 1, capacity + 1, capacity + 1)`, where `P[s, a, s2]` is the
probability that the next morning starts with `s2` units, and `R` of shape
`(capacity + 1, capacity + 1)`, where `R[s, a]` is the expected reward of the day.

In [ ]:
def build_model(capacity, demand_probs, price, unit_cost, order_fee, holding_cost):
    """Transition probabilities P[s, a, s2] and expected rewards R[s, a] of the shop."""
    # TODO
    raise NotImplementedError

In [ ]:
P, R = build_model(CAPACITY, DEMAND_PROBS, PRICE, UNIT_COST, ORDER_FEE, HOLDING_COST)
P, R = np.asarray(P, dtype=float), np.asarray(R, dtype=float)
assert P.shape == (6, 6, 6) and R.shape == (6, 6), f"P should have shape (6, 6, 6) and R shape (6, 6), got {P.shape} and {R.shape}"
assert np.allclose(P.sum(axis=2), 1), "each P[s, a] should be a probability distribution over the next stock, summing to 1"
assert np.allclose(np.triu(P[:, 0, :], k=1), 0), (
    "without an order the stock can only fall, but some P[s, 0] puts probability on a higher stock. "
    "A stock that went below zero, used as an index, wraps around to the end of the array"
)
assert np.allclose(P[2, 0], [0.7, 0.2, 0.1, 0, 0, 0]), f"P[2, 0] is {P[2, 0]}, expected [0.7, 0.2, 0.1, 0, 0, 0]"
assert np.allclose(P[3, 4], [0, 0.1, 0.3, 0.3, 0.2, 0.1]), f"P[3, 4] is {P[3, 4]}: ordering 4 onto 3 units fills the shelf to 5, and the other 2 are thrown away"
assert np.isclose(R[0, 0], 0), f"R[0, 0] is {R[0, 0]:.4f}: an empty shelf with no order sells nothing, pays nothing and holds nothing"
assert np.isclose(R[2, 0], 9.4), f"R[2, 0] is {R[2, 0]:.4f}, expected 9.4: units sold are min(stock, demand), and the holding cost is on the units left at the end of the day"
assert np.isclose(R[3, 0], 11.5), f"R[3, 0] is {R[3, 0]:.4f}, expected 11.5: the order fee is paid only on days with an order"
assert np.isclose(R[3, 4], -2.85), f"R[3, 4] is {R[3, 4]:.4f}, expected -2.85: all 4 units ordered are paid for, even the 2 that do not fit"
print("Part 1 passed")

## Part 2 (coding task): Value iteration

Write `value_iteration(P, R, gamma, tol)`. Start from $V = 0$ and apply the Bellman
optimality update to every state at once,

$$
V(s) \leftarrow \max_a \big( R[s, a] + \gamma\, P[s, a] \cdot V \big) ,
$$

where `P[s, a]` is the row of next-stock probabilities and the dot product with $V$ is the
expected value of the next morning. Repeat until the largest change in a sweep is below
`tol`, for at most 10,000 sweeps. Return the final `V` and the greedy policy: an integer
array giving, for each state, the action that attains the max.

The check runs your function on a two-state MDP first, then on the shop with `tol = 1e-10`.

In [ ]:
def value_iteration(P, R, gamma, tol):
    """Optimal values V[s] and the greedy policy, by repeated Bellman optimality sweeps."""
    # TODO
    raise NotImplementedError

In [ ]:
# A two-state MDP first. In state 0, action 0 pays 1 and moves to state 1, and action 1
# pays 0 and stays. In state 1, action 1 pays 2 and moves to state 0, and action 0 pays 0
# and stays.
P_toy = np.zeros((2, 2, 2))
P_toy[0, 0, 1] = P_toy[0, 1, 0] = P_toy[1, 0, 1] = P_toy[1, 1, 0] = 1.0
R_toy = np.array([[1.0, 0.0], [0.0, 2.0]])
V_toy, pi_toy = value_iteration(P_toy, R_toy, 0.5, 1e-12)
assert np.allclose(V_toy, [8 / 3, 10 / 3], atol=1e-8), (
    f"two-state MDP: V is {np.round(V_toy, 4)}, expected [2.6667, 3.3333]. "
    "Check which axis the max and the sum over next states use, and that gamma multiplies only the next state's value"
)
assert [int(a) for a in pi_toy] == [0, 1], f"two-state MDP: the policy is {pi_toy}, expected [0, 1], the greedy action in each state"

P, R = build_model(CAPACITY, DEMAND_PROBS, PRICE, UNIT_COST, ORDER_FEE, HOLDING_COST)
V, policy = value_iteration(P, R, GAMMA, 1e-10)
V, policy = np.asarray(V, dtype=float), [int(a) for a in policy]
Q = R + GAMMA * P @ V
gap = np.max(np.abs(Q.max(axis=1) - V))
assert gap < 1e-8, f"V does not satisfy the Bellman optimality equation: the largest gap is {gap:.2e}"
V_star = np.array([37.5440239382, 39.5440239382, 44.1435817471, 47.9544832917, 50.8312337164, 53.5440239382])
assert np.allclose(V, V_star, atol=1e-6), f"V is {np.round(V, 4)}, expected {np.round(V_star, 4)}"
taken = Q[np.arange(6), policy]
assert np.allclose(taken, V_star, atol=1e-6), (
    f"the policy is {policy} and earns {np.round(taken, 4)}, expected {np.round(V_star, 4)}: "
    "in each state, the order it names must attain the max"
)
for s in range(6):
    print(f"stock {s}: order {policy[s]}, value {V[s]:6.2f}")
print("Part 2 passed")

## Solution

### Part 1: The shop as arrays

In [ ]:
def build_model(capacity, demand_probs, price, unit_cost, order_fee, holding_cost):
    """Transition probabilities P[s, a, s2] and expected rewards R[s, a] of the shop."""
    n = capacity + 1
    P = np.zeros((n, n, n))
    R = np.zeros((n, n))
    for s in range(n):
        for a in range(n):
            stock = min(s + a, capacity)           # extra units are thrown away
            for d, p in enumerate(demand_probs):
                sold = min(stock, d)               # unmet demand is lost
                left = stock - sold
                P[s, a, left] += p
                R[s, a] += p * (price * sold - unit_cost * a
                                - order_fee * (a > 0) - holding_cost * left)
    return P, R

**Why it works.** Each demand `d` is one outcome of the day, with probability `demand_probs[d]`.
Clipping the stock at the capacity, selling `min(stock, d)` and keeping the rest gives the
next state, which is never negative and so is always a valid index. Writing
`left = stock - d` instead would give $-1$ for an unmet customer, and `P[s, a, -1]` silently
lands on the full-shelf state. The `+=` matters because several demands can leave the same
stock: with 2 units, demands of 2, 3 and 4 all empty the shelf, which is the 0.7 in `P[2, 0]`.
The expected reward is the same sum over demands, weighted by `p`.

### Part 2: Value iteration

In [ ]:
def value_iteration(P, R, gamma, tol):
    """Optimal values V[s] and the greedy policy, by repeated Bellman optimality sweeps."""
    V = np.zeros(P.shape[0])
    for _ in range(10_000):
        Q = R + gamma * P @ V                      # Q[s, a]: one step, then V
        V_new = Q.max(axis=1)
        if np.max(np.abs(V_new - V)) < tol:
            break
        V = V_new
    return V_new, Q.argmax(axis=1)

**Why it works.** `P @ V` has shape `(6, 6)`, and entry `[s, a]` is the expected value of the
next morning's stock. Adding `R` and taking the max over axis 1 is one Bellman sweep for all
states at once. The sweep is a $\gamma$-contraction, so each sweep's largest change is at most 0.9
times the one before, and the loop ends.

The policy reorders only when 0 or 1 unit is left, and then fills the shelf. The fixed fee
makes small orders expensive, so the shop orders seldom and orders a lot. The values agree:
$V(0) = 37.54$ is exactly $V(5) - 16$, because ordering 5 units, for 10 plus the fee of 6,
turns an empty shelf into a full one before any customer arrives. In the same way
$V(1) = V(5) - 14$. With 2 units, filling the shelf would cost 12 and be worth
$53.54 - 12 = 41.54$, less than the $44.14$ of waiting a day.